# Day 40 — Data Governance, Personal Data, and Snowflake


Data engineering moves and transforms data. Data governance determines why that data exists, who is responsible for it, who may use it, and what happens when it is no longer needed.

We follow **Northstar Shop**, a fictional retailer using Snowflake for regional reporting and customer support. All customer records, organizational tags, policies, and retention periods in examples are invented. Example policies are proposals, not Snowflake defaults or legal requirements.

By the end, you should understand personally identifiable information, table and column tags, masking, row filtering, privacy obligations, and the complete data lifecycle.

**Disclaimer: The topics presented in this notebook should be taken with a grain of salt. The trainer is not a policy or data governance expert, and the material is intended for educational and discussion purposes only. You must consult internal expert**

## Learning route

1. Governance foundations and responsibilities.
2. Personal information and identification risk.
3. Snowflake and access control.
4. Classification, tagging, masking, and row filtering.
5. Network addresses and payment-card protection.
6. European privacy law and individual rights.
7. Aging data, retention, deletion, and recovery.
8. Operational controls, a case study, exercises, and glossary.

Read in order initially. Each technical topic starts with an introduction; abbreviations are expanded on first use and collected in the glossary.

## 1. What is data governance?

**Data governance** is the system of decisions, responsibilities, standards, and checks used to manage data throughout its life. It answers:

- What does this field mean, and which source should we trust?
- Why are we collecting it?
- Who approves access and resolves quality problems?
- Which uses are permitted?
- When should we delete it, and how will we verify completion?

**Data management** is the daily implementation: loading records, correcting errors, granting permissions, and deleting expired data. **Data security** protects confidentiality, integrity, and availability. **Data privacy** concerns appropriate processing of information about people. **Compliance** means meeting applicable laws, contracts, and standards.

Encrypted data can still be used for an inappropriate purpose. Equally, excellent privacy documentation cannot prevent exposure through an overly broad role. Governance connects the business decision, the technical control, and evidence that the control works.

Northstar's decision might be: “Regional analysts may study their assigned region's sales, but do not need customer contact details.” Engineers translate that into object permissions, row restrictions, column protection, and reviews.

## 2. Governance covers quality, meaning, and ownership

A **data asset** is a useful collection of information, such as a customer table. A **data catalog** is an organized inventory of assets and their descriptions. **Metadata** is information about data: its owner, source, meaning, sensitivity, and retention rule.

| Area | Question | Example |
|---|---|---|
| Ownership | Who decides? | Customer Operations approves customer-data use |
| Definitions | Do teams mean the same thing? | Completed orders exclude canceled orders |
| Quality | Is data fit for use? | Region values must match an approved list |
| Privacy | Is the use appropriate? | Addresses support delivery |
| Security | Who can access what? | Analysts cannot see full contact details |
| Lifecycle | How long is data required? | Temporary import files expire |
| Lineage | Where did data go? | Order events feed sales reports |
| Accountability | Can we demonstrate control? | Access approvals are recorded |

**Data lineage** describes movement and transformation between sources and outputs. It helps find reports affected by an incorrect field and copies affected by an erasure request.

**Data quality** means fitness for the intended use. Wrong customer references can cause a rights request to miss records. Wrong region values can cause incorrect access decisions. Quality therefore supports privacy and security, not just accurate reporting.

## 3. People and responsibilities

| Responsibility | Typical work |
|---|---|
| Data owner | Approves purpose, access, and retention |
| Data steward | Maintains definitions, classifications, and quality rules |
| Data engineer | Implements pipelines, protection, and deletion workflows |
| Security team | Manages identity, privileged access, and incidents |
| Privacy and legal teams | Interpret obligations and rights requests |
| Data consumer | Uses information only for approved work |

**Separation of duties** divides sensitive responsibilities. For example, one person approves access while another maintains protection rules. It reduces the opportunity to approve one's own access and silently remove controls.

A business data owner is an organizational responsibility. Snowflake object ownership is a technical privilege held by a role. The two should be coordinated, but they are not interchangeable.

**Least privilege** means providing only the access required for the job. Access should also expire or be reviewed when the person's job changes.

## 4. Personally identifiable information

**Personally Identifiable Information (PII)** can identify a person directly or through combination with other information. Definitions vary by framework. The **General Data Protection Regulation (GDPR)** uses **personal data**: information relating to an identified or identifiable natural person. A natural person is a human being.

| Type | Examples | Risk |
|---|---|---|
| Direct identifier | Personal email, national identifier | Points toward a person |
| Indirect identifier | Age, postal area, job title | Combinations can identify someone |
| Online identifier | Network address, device identifier | Connects activity to a person |
| Linkable internal identifier | Customer key | Another table can reveal identity |
| Personal activity | Purchases, location history | Describes an identifiable person |

An **Internet Protocol (IP) address** is used in network communication. It may be personal data where a person can be identified from it and other available information. A numeric identifier is not automatically anonymous.

A customer key remains important even after names are removed: a separate account table may reconnect purchases to people. Review relationships, not just column names. [GDPR Article 4 and Recital 30](https://eur-lex.europa.eu/eli/reg/2016/679/oj/eng).

## 5. Sensitive data and re-identification

**Sensitive data** is information whose exposure or misuse could cause harm. It includes personal information, credentials, and business secrets. Organizational sensitivity levels are handling labels, not universal legal categories.

Northstar could use Public, Internal, Confidential, and Restricted. Define what each label requires: approved roles, export restrictions, retention rules, and review frequency.

GDPR **special categories** include health, racial or ethnic origin, political opinions, religious beliefs, trade-union membership, genetic information, certain biometric information used for unique identification, and sex-life or sexual-orientation information. Additional processing conditions apply. Financial data is sensitive in practice but is not automatically an Article 9 special category. [GDPR Article 9](https://eur-lex.europa.eu/eli/reg/2016/679/oj/eng).

**Re-identification** reconnects a supposedly de-identified record to a person. A report containing branch, age band, and job title can identify the only senior accountant at a small branch.

Ask whether records can be joined to another dataset, whether a group contains very few people, and whether repeated reports can reveal an individual's value through subtraction.

## 6. Snowflake concepts

**Snowflake** is a cloud data platform used to store, process, and analyze information.

| Concept | Introduction |
|---|---|
| Account | Administrative environment containing users and objects |
| Database | Container for schemas |
| Schema | Namespace grouping tables, views, and policies |
| Table | Stored records organized in rows and columns |
| Row | One record, such as one customer |
| Column | One attribute, such as email |
| View | Named definition presenting underlying data |
| Virtual warehouse | Compute resources that perform work |
| Role | Collection of access privileges |
| Policy | Reusable rule evaluated to control behavior |

**Snowsight** is Snowflake's web interface. **Snowflake Horizon Catalog** brings together capabilities for understanding and governing data. Discovery and description need to be connected to configured enforcement. [Snowflake governance overview](https://docs.snowflake.com/en/guides-overview-govern).

The organization supplies the business purpose and approved rules. Snowflake provides technical mechanisms to implement parts of that design; using the platform does not automatically establish compliance.

## 7. Access control comes first

**Authentication** verifies who is connecting. **Authorization** determines what that identity may do. **Role-Based Access Control (RBAC)** organizes privileges through roles. Role hierarchies can make privileges available through inheritance.

Snowflake object privileges establish whether a user can access a table. Masking and row access policies further constrain permitted access; they do not grant table access themselves. Review the complete effective role context. [Snowflake access control](https://docs.snowflake.com/en/user-guide/security-access-control-overview).

An **entitlement** is an approved allowance, such as access to West-region records. Record its approver, purpose, and review or expiry date.

**Multi-Factor Authentication (MFA)** uses multiple authentication factors. **Single Sign-On (SSO)** connects access to a shared identity system. These protect identity but do not decide which customer records are appropriate for a particular task.

Northstar should use purpose-specific analyst, support, and reconciliation roles rather than routine administrator access.

## 8. Classification: discovering sensitive columns

**Data classification** assigns categories based on meaning or sensitivity. Manual classification uses human review; automated classification uses patterns and other signals.

Snowflake sensitive data classification can discover supported sensitive categories and apply classification tags. `SNOWFLAKE.CORE.SEMANTIC_CATEGORY` describes the kind of information; `SNOWFLAKE.CORE.PRIVACY_CATEGORY` describes its privacy category. Configured workflows can map classification results to user-defined tags. [Snowflake classification](https://docs.snowflake.com/en/user-guide/classify-intro).

A suggested process is inventory, classify, review uncertain results, apply organizational tags, connect protection, and recheck changes.

A **false positive** labels harmless data as sensitive. A **false negative** misses sensitive information. A column named `comments` might contain phone numbers even if no category is detected. Names and samples do not guarantee complete coverage.

Northstar should restrict newly arrived datasets until review is complete. That release restriction is an organizational pipeline design, not an automatic consequence of running classification.

## 9. PII tagging of tables and columns

A **tag** attaches descriptive metadata to an object. Snowflake user-defined tags are schema-level objects and can be assigned to supported objects, including tables and columns. Tags help inventory and governance; a tag by itself does not hide information. [Snowflake object tagging](https://docs.snowflake.com/en/user-guide/object-tagging/introduction).

These are proposed organizational labels, not built-in tag names:

| Scope | Tag and value | Meaning |
|---|---|---|
| Customer table | `contains_personal_data = yes` | Dataset includes personal information |
| Customer table | `business_owner = customer_operations` | Responsible team |
| Customer table | `retention_class = customer_lifecycle` | Applicable documented lifecycle rule |
| Email column | `data_category = email` | Meaning of this field |
| Email column | `sensitivity = restricted` | Handling requirement |
| Payment token column | `data_category = payment_token` | Payment-reference purpose |
| Region column | `data_category = sales_region` | Business attribute |

A table tag provides a broad signal. Column tags describe individual fields precisely. A table can contain both restricted emails and ordinary order totals.

Never place real customer information in tag values or descriptions. Metadata can be discoverable too.

## 10. Inheritance and propagation

**Tag inheritance** allows tags higher in Snowflake's object hierarchy to apply to supported child objects. A table tag can affect columns. More specific assignments can change effective values under inheritance rules. Inspect effective tags, not only direct assignments. [Snowflake tag inheritance](https://docs.snowflake.com/en/user-guide/object-tagging/inheritance).

**Tag propagation** transfers tags through supported data movement or dependency relationships. Snowflake supports configurable automatic propagation for user-defined tags, subject to supported operations and conflict rules. It differs from inheritance; do not assume every derived table or external export receives the original tags. [Snowflake tag propagation](https://docs.snowflake.com/en/user-guide/object-tagging/propagation).

For Northstar, a broad table label is useful for discovery. A masking-enabled tag applied broadly can also affect compatible child columns. That may conceal ordinary text fields as well as emails, so test the actual outcome.

When birth date becomes an age band, review the transformed field's meaning and risk. When email is copied into notes, the sensitivity has not disappeared.

## 11. Connecting tags to masking

**Tag-based masking** connects descriptive metadata to enforcement. Snowflake can associate a masking policy with a tag. When that tag applies to a column, a compatible policy can protect it. A tag can have policies for different supported data types; the column and policy types must be compatible.

A directly assigned column masking policy takes precedence over tag-based masking. Inspect exceptions and effective coverage. [Snowflake tag-based masking](https://docs.snowflake.com/en/user-guide/tag-based-masking-policies).

Northstar's conceptual workflow is:

**Identify contact field → assign contact-data tag → connect policy → test permitted roles → monitor coverage.**

The owner approves who may see original values. Engineers implement the rule and attachment. Stewards review classifications. Security reviewers inspect privileged changes.

A `retention_class` tag does not delete records by itself. It needs a lifecycle process that interprets the label. Likewise, a tag saying `restricted` is not proof of restricted access unless enforcement exists.

## 12. Data masking in depth

**Data masking** conceals or substitutes values to reduce exposure. **Dynamic masking** does this during a query. **Static masking** creates a transformed copy, often for testing.

Snowflake Dynamic Data Masking uses schema-level policies on supported columns. Conditions and execution context determine whether the consumer receives original, partially concealed, or replacement values. It does not overwrite the stored original. The feature requires Enterprise Edition or higher. [Snowflake Dynamic Data Masking](https://docs.snowflake.com/en/user-guide/security-column-ddm-intro).

| Field | Proposed analyst display | Proposed approved support display |
|---|---|---|
| Email | `[hidden]` | Original only when needed |
| Phone | `[hidden]` | Limited digits for verification |
| Birth date | Reviewed age band in a reporting field | Hidden unless required |
| Card reference | Last four digits if needed | Last four digits if needed |

A partial mask reveals information intentionally. Even a concealed email prefix can identify someone in a small group. Choose the least revealing useful representation.

Masking is not deletion or anonymization. If an authorized consumer exports original values, the resulting copy needs independent protection.

## 13. Related protection techniques

| Technique | Introduction | Limitation |
|---|---|---|
| Encryption | Uses cryptographic keys to convert readable data into ciphertext | Authorized decryption recovers the original |
| Hashing | Produces a digest from an input | Predictable inputs can be guessed; stable hashes permit linking |
| Tokenization | Replaces a sensitive value with a managed substitute | A mapping or recovery service may reconnect identity |
| Pseudonymization | Separates attribution from data using protected additional information | Data remains personal when attribution remains possible |
| Anonymization | Makes identification no longer reasonably possible | Requires contextual assessment, not just removing names |
| Aggregation | Combines records into group statistics | Small groups and repeated comparisons can reveal individuals |

A plain hash of an email or IP address is not evidence of anonymity. An attacker can calculate hashes of likely inputs. A **salt** is additional input used in hashing; adding one does not automatically establish anonymity.

**Synthetic data** consists of artificially created records. Prefer it for intern exercises. If generated from private source data, review whether it can reproduce sensitive examples.

Choose controls according to the task. Encryption protects stored or transmitted information; masking controls a consumer's view; tokenization can reduce exposure of original identifiers. These protections can be combined.

## 14. Row filtering in Snowflake

**Row filtering**, also called **Row-Level Security (RLS)**, limits visible records. Snowflake implements it through row access policies on supported tables and views.

A policy evaluates conditions using row values and execution context and returns a true-or-false result. It can consult an entitlement mapping table. Policy-owner privileges support that lookup without granting consumers direct access to the mapping table. Row access policies require Enterprise Edition or higher. [Snowflake row access policies](https://docs.snowflake.com/en/user-guide/security-row-intro).

Northstar's proposed rule is: allow a regional record only if its region appears in the user's approved entitlement list. Missing entitlements should provide no regional access.

| Record | Region | West analyst | East analyst |
|---|---|---|---|
| C101 | West | Visible | Hidden |
| C102 | East | Hidden | Visible |
| C103 | West | Visible | Hidden |

Hidden records remain stored. A dashboard's optional region selector is not a substitute for an enforced database restriction.

## 15. Combining row filtering and masking

Row filtering answers “Which records?” Masking answers “Which values?” When both apply to the same object, Snowflake evaluates the row access policy before masking policies. [Snowflake policy evaluation](https://docs.snowflake.com/en/user-guide/security-row-intro).

A conceptual request walkthrough is:

1. Authenticate the user and establish the role context.
2. Check object privileges.
3. Apply row eligibility rules.
4. Apply protected column representations.
5. Return the permitted result and collect monitoring evidence.

This is a teaching sequence, not a complete query-optimizer description.

| West analyst result | Region | Email | Order amount |
|---|---|---|---|
| C101 | West | `[hidden]` | 1,200 |
| C103 | West | `[hidden]` | 800 |

The East record is absent; West emails are concealed; useful amounts remain. Review alternate tables and applications exposing the same data. Role names alone do not enforce business purpose.

## 16. IP address data and privacy

An Internet Protocol address routes network traffic. Services may record it for security investigations, rate limiting, or approximate geographic analysis. Shared addresses can represent many users; changing addresses can still be linked using timestamps and other records.

Identifiability determines whether an address is personal data in context. European case law recognizes identification through additional information. [European Parliament response on IP addresses](https://www.europarl.europa.eu/RegData/questions/reponses_qe/2024/002546/P10_RE%282024%29002546_EN.pdf).

**Internet Protocol version 4 (IPv4)** uses 32-bit addresses. **Internet Protocol version 6 (IPv6)** uses 128-bit addresses. A masking transformation designed for dotted IPv4 text may not handle IPv6 correctly.

Northstar's security team may need restricted detailed events for an approved period. Marketing may need only reviewed country-level totals. Interns can use invented addresses.

Removing part of an address reduces precision but does not guarantee anonymity. Exact timestamps, account keys, and device characteristics can preserve linkage. Avoid unnecessary copies in debug logs, support tickets, and exports.

## 17. Credit-card masking and payment data

The **Primary Account Number (PAN)** is the payment-card number. The **Payment Card Industry Data Security Standard (PCI DSS)** is an industry standard for protecting payment-account data. It is distinct from GDPR; both can matter to one system.

A fictional support display might show `**** **** **** 1234`. Last-four display is an example design, not a complete compliance solution. PCI DSS distinguishes display masking from protection of the stored PAN. A hidden screen value does not make a stored full number unreadable. [PCI display masking](https://www.pcisecuritystandards.org/faqs/1071/); [masking versus truncation](https://www.pcisecuritystandards.org/faqs/1146/).

**Card Verification Value (CVV)** and **Card Verification Code (CVC)** are names for card verification codes. In Northstar's merchant workflow, they must not be retained after authorization, even encrypted. Masking them does not fix prohibited storage. [PCI authentication-data guidance](https://www.pcisecuritystandards.org/faqs/1533/).

A proposed architecture sends card details to an approved payment provider and places only necessary tokens, transaction references, and last four digits in analytics. Tokenization does not automatically remove all systems from PCI DSS scope; assess the actual design and recovery capabilities.

## 18. GDPR introduction, scope, and lawful basis

The General Data Protection Regulation is the **European Union (EU)** data-protection regulation. It applies to relevant processing in the context of EU establishments and can also cover organizations outside the EU offering goods or services to, or monitoring behavior of, people in the EU. Scope is not simply a citizenship test.

A **data subject** is the person concerned. A **controller** decides why and how processing occurs. A **processor** processes on the controller's behalf. Contracts and actual activities determine responsibilities.

The six lawful bases are consent, contract, legal obligation, vital interests, public task, and legitimate interests. Each has conditions. Consent is not required for every operation. Special-category data requires additional conditions. [European Commission: GDPR application](https://commission.europa.eu/law/law-topic/data-protection/information-business-and-organisations/application-gdpr_en).

Northstar collecting an email to fulfill an order does not automatically justify every future marketing use. Engineers need the approved purpose and use rules, rather than infer permission from availability.

## 19. What GDPR restricts

| Principle | Meaning | Engineering implication |
|---|---|---|
| Lawfulness, fairness, transparency | Justified and understandable processing | Document purpose and approved use |
| Purpose limitation | Avoid incompatible reuse | Review delivery-data reuse for marketing |
| Data minimization | Use only necessary information | Exclude birth date from ordinary sales reporting |
| Accuracy | Keep relevant information correct | Propagate approved corrections |
| Storage limitation | Avoid unjustified indefinite retention | Implement expiry and reviews |
| Integrity and confidentiality | Protect against unauthorized use and loss | Configure protection and access controls |
| Accountability | Demonstrate compliance | Preserve decisions and evidence |

There is no universal GDPR retention period for every dataset. Privacy by design and by default means considering protection during design and starting with limited necessary processing. [European Commission: GDPR principles](https://commission.europa.eu/law/law-topic/data-protection/information-business-and-organisations/principles-gdpr_en).

International transfers need separate assessment and applicable safeguards. A European storage region alone does not settle every transfer issue, including remote access or replication. [GDPR Chapter V](https://eur-lex.europa.eu/eli/reg/2016/679/oj/eng).

## 20. Individual rights and engineering workflows

GDPR rights include information, access, correction, erasure in applicable circumstances, restriction, portability where applicable, and objection. Protections also concern certain solely automated decisions with legal or similarly significant effects. Rights have conditions and exceptions; erasure is not unconditional. [European Commission: individual rights](https://commission.europa.eu/law/law-topic/data-protection/information-individuals_en).

A **Data Subject Access Request (DSAR)** exercises access rights. Other requests concern erasure or correction. A **Data Protection Officer (DPO)** advises and monitors data-protection compliance where appointed or required.

Northstar's proposed workflow:

1. Route the request through the privacy process and verify identity proportionately.
2. Determine scope and applicable exceptions.
3. Find linked records using controlled identity matching and lineage.
4. Apply the approved action across relevant systems.
5. Record completion and justified retention exceptions.
6. Prevent inappropriate recreation through future source reloads.

An identity mapping helps locate records but needs protection itself. A masked email remains stored and does not satisfy an approved erasure action.

## 21. Data aging and retention

**Data aging** is the change in usefulness, accuracy, risk, and obligations over time. An old address may be incorrect; an old event may no longer justify detailed storage. Old information can remain sensitive.

**Retention** decides how long information is kept. A **retention schedule** records purpose, starting event, period, owner, exceptions, and disposal action. **Archiving** moves data into less frequently used storage; it remains retention.

These periods are fictional classroom examples, not legal requirements:

| Dataset | Starting event | Example rule | End action |
|---|---|---|---|
| Temporary import file | Verified ingestion | 7 days | Delete staging copy |
| Security event detail | Event time | 90 days unless scoped investigation hold | Delete expired detail |
| Customer profile | Account closure | Review after 30 days | Remove unnecessary fields |
| Invoice | Accounting period end | Applicable legal period | Restrict to required use, then dispose |
| Regional summary | Report creation | Annual usefulness and privacy review | Retain or retire after assessment |

Choose the correct starting event: account creation and account closure give different expiry dates.

A **legal hold** suspends disposal of records needed for a legal matter. Record its scope, approver, restrictions, review, and release process. It should not indefinitely preserve unrelated information.

## 22. Snowflake retention, Time Travel, and Fail-safe

Deleting current records differs from expiry of recovery copies.

**Time Travel** allows access to or recovery of historical Snowflake data within a configured period. For relevant native tables, standard retention is generally one day; eligible permanent objects on Enterprise Edition or higher support up to 90 days. Settings and object type matter. Historical retention does not automatically delete aged business rows. [Snowflake Time Travel](https://docs.snowflake.com/en/user-guide/data-time-travel).

**Fail-safe** is a separate non-configurable seven-day recovery period for permanent-table historical data after Time Travel. Snowflake operates it for recovery; it is not a normal user archive. Transient and temporary tables have no Fail-safe. [Snowflake Fail-safe](https://docs.snowflake.com/en/user-guide/data-failsafe).

A **transient table** has reduced recovery protection. A **temporary table** generally exists for a session. Choose these types based on privacy and recovery requirements together.

A **clone** is another object created from existing data; it can preserve information beyond changes to the original. Inventory current tables, historical retention, clones, replicas, source systems, staged files, exports, and consumer copies. Do not promise immediate removal of every historical copy after deleting current rows.

## 23. Implementing the lifecycle

A **pipeline** automates data movement or transformation. An **orchestrator** coordinates steps, schedules, and failures. Snowflake tasks or external orchestration can participate; an organization still defines the rule and covers external destinations.

1. **Collect:** document purpose and minimize fields.
2. **Ingest:** restrict new data, assign ownership, and record source.
3. **Release:** classify, tag, attach protection, and test roles.
4. **Use:** monitor quality, access, and new copies.
5. **Age:** calculate eligibility from the right event and rule.
6. **Review:** check scoped holds and required retention.
7. **Dispose:** address in-scope copies and record results.
8. **Verify:** check completion and prevent inappropriate re-ingestion.

An **idempotent** operation can be retried without producing an incorrect extra effect. Deletion workflows should tolerate retries. If warehouse cleanup succeeds but export cleanup fails, record the export as pending rather than declaring the whole request complete.

A blank retention date should trigger review rather than accidental indefinite storage. A tag describing retention needs an actual process that acts on it.

## 24. Auditing and incidents

An **audit trail** records relevant actions and decisions. Snowflake Access History provides information about supported access and modifications, including object and column relationships. It requires Enterprise Edition or higher. Review coverage and latency; it does not instantly describe every external application action. Query history and policy inventories provide complementary evidence. [Snowflake Access History](https://docs.snowflake.com/en/user-guide/access-history).

Maintain approvals, policy changes, entitlement reviews, coverage checks, deletion outcomes, and incident records. A query log does not prove a valid business purpose.

A **personal data breach** can involve unauthorized disclosure, alteration, loss, or unavailability. Qualifying GDPR breaches require supervisory-authority notification without undue delay and, where feasible, within 72 hours of awareness, subject to the risk exception. High-risk breaches may also require notifying individuals. Interns should immediately follow the incident process. [European Commission: obligations](https://commission.europa.eu/law/law-topic/data-protection/information-business-and-organisations/obligations_en).

Preserve necessary evidence through approved channels. Do not duplicate exposed personal information in a broadly accessible support ticket.

## 25. Downstream tools and changing data

**Schema drift** is a structural change, such as a new phone column. It can introduce personal information after the original review. Add release checks for new fields and changed meanings.

**Business Intelligence (BI)** tools produce dashboards and reports. They may connect through a shared service identity rather than each viewer's identity. Inspect whose context Snowflake evaluates and how application users are restricted.

An **export** creates a copy outside the protected access path. Snowflake policies do not automatically follow original values into spreadsheets or external systems. Govern destination users, storage, retention, and deletion.

Review whether a report needs individual records at all. Aggregates may be sufficient, but small groups can still reveal information. A secure source table does not establish the safety of every downstream copy.

## 26. Complete Northstar design

Northstar stores customer reference, name, email, region, order amount, payment token, card last four digits, account closure date, and login IP address.

| Decision | Proposed implementation |
|---|---|
| Ownership | Customer Operations owns contacts; Security owns detailed login events |
| Minimization | Analytics receives necessary payment references, not verification codes |
| Classification | Table tags describe dataset purpose; column tags identify sensitive fields |
| Object access | Purpose-specific roles receive necessary privileges |
| Enforcement | Regional row restrictions combine with field-specific masking |
| Lifecycle | Scheduled reviews, deletion workflows, and copy inventories |
| Evidence | Access approvals, role tests, coverage checks, and disposal records |

| Purpose | Rows | Contact data | Payment data | Login IP |
|---|---|---|---|---|
| West sales analysis | West only | Hidden | Excluded unless needed | Excluded |
| Customer support | Assigned customer scope | Minimum needed | Last four if needed | Hidden |
| Reconciliation | Approved transactions | Hidden unless justified | Approved token and reference | Excluded |
| Security investigation | Approved investigation scope | Limited supporting context | Excluded unless justified | Detailed if approved |
| Intern training | Synthetic records | Synthetic | Synthetic | Invented |

These are reviewable requirements, not automatic platform defaults. Owners approve them; engineers implement and validate actual behavior.

## 27. Validation before release

A **positive test** checks that approved work succeeds. A **negative test** checks that prohibited access fails. Use synthetic records with known expected results.

| Situation | Expected behavior |
|---|---|
| West analyst reads customer data | Only West records, protected contact values |
| No regional entitlement | No regional customer access |
| Entitlement expires | Access removed through the approved process |
| New sensitive column arrives | Restricted until reviewed and protected |
| Derived table is added | Effective tags and policies inspected and tested |
| Alternate access path exists | Same intended business restrictions hold |
| Approved erasure completes | Current in-scope records removed; copy and recovery handling recorded |
| Source reload occurs | No inappropriate recreation of erased records |

The release restriction is a proposed pipeline control, not an automatic classification feature. Include missing region values, multiple regions, service identities, inherited roles, and privileged policy changes in review.

## 28. Common misconceptions

| Misconception | Correction |
|---|---|
| A PII tag hides a column | Configured policies enforce protection |
| A table tag precisely describes every field | Columns need meaningful classification |
| Masked means deleted | Originals may remain stored and recoverable |
| Hashed means anonymous | Guessing and linking may remain possible |
| Row filtering protects every value | Visible rows can contain sensitive fields |
| Encryption makes every use acceptable | Purpose, rights, and retention still matter |
| GDPR always requires consent | Multiple lawful bases exist with conditions |
| GDPR always requires immediate erasure | Applicable exceptions require assessment |
| Old data is harmless | Sensitivity can persist |
| Platform certification makes our design compliant | Organizations retain implementation obligations |
| Administrator access establishes purpose | Technical power differs from approved need |
| Card display masking solves payment compliance | Stored data needs separate protection |

## 29. Discussion exercises

1. A table contains customer keys and purchases but no names. Why might it remain personal data?
2. A table is tagged as containing PII, yet emails are visible. What must be inspected?
3. A West analyst sees only West rows but full phone numbers. Which control is missing?
4. A team hashes IP addresses and retains exact timestamps indefinitely. What should reviewers challenge?
5. Support needs to identify which card was used. What minimum information might suffice?
6. Erasure is requested, but some invoices have a legal retention obligation. How should purposes be separated?
7. Why must a new free-text field be reviewed?
8. Why can a source reload undermine deletion?
9. Why does a BI tool's shared identity matter?
10. Why is current-row deletion different from immediate erasure of all historical copies?

## 30. Suggested answers

1. Keys can reconnect activity to people through other datasets.
2. Check policy attachment, type compatibility, precedence, privileges, and effective role behavior.
3. Appropriate column protection, such as masking under the approved rule.
4. Purpose, retention justification, guessing and linkage risk, and whether less detail is sufficient.
5. An approved transaction reference and last four digits may suffice; merchant verification codes must not be retained after authorization.
6. Apply approved erasure scope, isolate legitimately retained records, restrict further use, and document expiry.
7. It can introduce personal information missed by the earlier classification.
8. Upstream data and approved prevention mechanisms must be coordinated with deletion.
9. Snowflake may evaluate the shared identity rather than the individual viewer.
10. Recovery periods, clones, replicas, exports, and source copies have separate lifecycle behavior.

## 31. Abbreviation glossary

| Abbreviation | Expansion |
|---|---|
| PII | Personally Identifiable Information |
| GDPR | General Data Protection Regulation |
| EU | European Union |
| IP | Internet Protocol |
| IPv4 | Internet Protocol version 4 |
| IPv6 | Internet Protocol version 6 |
| RBAC | Role-Based Access Control |
| RLS | Row-Level Security |
| MFA | Multi-Factor Authentication |
| SSO | Single Sign-On |
| PAN | Primary Account Number |
| PCI DSS | Payment Card Industry Data Security Standard |
| CVV | Card Verification Value |
| CVC | Card Verification Code |
| DSAR | Data Subject Access Request |
| DPO | Data Protection Officer |
| BI | Business Intelligence |

The official references are linked beside the relevant explanations throughout this notebook. Example retention periods and access designs are classroom proposals. In a real project, the data owner and privacy team define applicable requirements, while engineers verify effective controls and feature availability in the target account.